# What is LoRA and QLoRA?
**LoRA (Low-Rank Adaptation)** is a method for fine-tuning large models by introducing low-rank matrices into pre-trained weights, significantly reducing the number of trainable parameters while maintaining performance.

**QLoRA (Quantized LoRA)** builds on this by incorporating quantization, which uses lower-precision data types to further reduce memory usage and computational requirements.

While both techniques aim to make fine-tuning more efficient, QLoRA is particularly advantageous for resource-constrained environments, as it enables fine-tuning on smaller hardware without sacrificing model accuracy.

## 📌 Why Fine-Tune a Model for Customer Support?
Large language models (LLMs) are powerful, but fine-tuning them on domain-specific data makes them more accurate, efficient, and cost-effective. 

In this project, we:

✅ Use 4-bit quantization to **reduce memory usage**, making it possible to fine-tune large models on limited hardware.

✅ Implement LoRA to efficiently adapt the model to new data without retraining all parameters.

✅ Fine-tune the model on a customer support FAQs dataset **to enhance its ability** to respond to common user queries.

# 📂 Setup

In [ ]:
%pip install -qU transformers accelerate bitsandbytes trl datasets peft tokenizers

## Import libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from trl import SFTTrainer, TrainingArguments

: 

## 𝌭 Model preparation

### Model config

In [ ]:
class CFG:
    model = "Qwen/Qwen2.5-3B-Instruct"

### Quantization config

Quantization reduces the model’s memory footprint, allowing it to run efficiently on standard hardware. Here’s how we configure it:

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Load model in 4-bit precision
    bnb_4bit_quant_type='nf4',  # NormalFloat4 quantization for better accuracy
    bnb_4bit_compute_dtype=torch.float16,  # Compute using float16
    bnb_4bit_use_double_quant=True  # Enable double quantization for numerical stability
)

### Model creation

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    CFG.model,
    device_map="auto",
    quantization_config=bnb_config,
)

# 📖 Tokenization and Dataset Preparation

## Tokenization

We need to tokenize the dataset correctly before training:

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Ensures padded tokens don’t affect training
tokenizer.padding_side = "right"  # Ensures alignment with causal models

Many models don’t have a default pad token, so we set it to the EOS token to prevent issues when batching sequences.

Setting the padding to the right allows causal models to predict the next token correctly without looking ahead.

## 📌 Load and Preprocess the Dataset

In [ ]:
dataset = load_dataset("Victorano/customer-support-1k", split="train")
dataset[0]

In [ ]:
dataset = dataset.remove_columns(['flags', 'category','intent','text'])
dataset = dataset.train_test_split(test_size=0.2)

We structure each training example as a chat format:

In [ ]:
instruction = """You are a helpful customer support bot..."""
def template(row):
    row_json = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]}
    ]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row

dataset = dataset.map(template, num_proc=4)

> By formatting our data as system-user-assistant exchanges, we provide better conversational flow for training.

# 🔄 Fine-Tuning with QLoRA

**LoRA (Low-Rank Adaptation)** allows us to fine-tune LLMs efficiently by adding small trainable layers instead of modifying all parameters.

In [ ]:
lora_config = LoraConfig(
    r=4,  # Rank for low-rank matrices
    lora_alpha=8,  # Scaling factor
    lora_dropout=0.2,  # Regularization
    task_type="CAUSAL_LM"  # Language modeling task
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Now, setup training arguments:

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",  # Directory where model checkpoints and logs will be saved
    num_train_epochs=3,  # Number of training epochs (full passes through the dataset)
    per_device_train_batch_size=1,  # Batch size for training per device (GPU/CPU)
    per_device_eval_batch_size=1,  # Batch size for evaluation per device
    warmup_steps=5,  # Number of warmup steps for learning rate scheduler
    learning_rate=2e-4,  # Initial learning rate for optimizer
    fp16=True,  # Enable mixed precision training (float16) for performance optimization
    report_to="none",  # Disable reporting to tracking tools like TensorBoard or Weights & Biases
)

Start training using SFTTrainer:

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    args=training_arguments,
    peft_config=lora_config,
)

trainer.train()

# 📢 Generating Responses

Test the fine-tuned model:

In [ ]:
def generate(input_prompt, model):
    messages = [{"role": "system", "content": instruction},
                {"role": "user", "content": input_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=2048)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate("Where can I see payment options?", model))

# 💾 Save and Reload the Model

Save the trained model:

In [ ]:
model.save_pretrained("./customer-faq-qwen")

# ␡ Delete model from memory

In [ ]:
import gc

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()